# MHW revision: all new analyses and corrected figures

This notebook reruns every analysis and figure added during the revision of the manuscript
*Causal persistence benchmarks for 1–14-day machine-learning forecasts of marine heatwaves in the northern Bay of Bengal*.

| Step | What it makes | Manuscript item |
|---|---|---|
| 1 | RONI previous-winter test | Table S11 (RONI rows) |
| 2 | Label used as a model input | Table S14, Section 3.2 |
| 3 | Past-only vs full-record design, detrended and raw labels | Table 6, Table S13 |
| 4 | Figs 2, 4, 6, 7 and S3 | from `results/tables` of the GitHub repository |
| 5 | Fig. S2 and Table S7 (case studies) | from your Drive `processed` folder |

**How to run:** Runtime → Run all. Steps 2 and 3 refit the model many times; on a 2-core machine each took about 5 minutes, so the whole notebook should finish in roughly 15 minutes.
Their results are saved to Google Drive after each lead, so if Colab disconnects, run all again and finished leads are skipped.
Step 6 compares your numbers with the values in the manuscript.


## 0. Setup: Drive, repository, output folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess

PROC = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/processed'   # your existing processed data
REV  = '/content/drive/MyDrive/Colab Notebooks/MHW_Project_Data/revision'    # new: all revision outputs
REPO = '/content/mhw-bay-of-bengal-forecast'
os.makedirs(REV, exist_ok=True)

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/abu2556/mhw-bay-of-bengal-forecast', REPO], check=True)
os.chdir(REPO)

# long runs write straight to Drive, so nothing is lost if the runtime disconnects
for sub in ('past_only', 'label_input'):
    os.makedirs(f'{REV}/{sub}', exist_ok=True)
    link = f'results/{sub}'
    if not os.path.islink(link):
        if os.path.exists(link): subprocess.run(['rm', '-rf', link])
        os.symlink(f'{REV}/{sub}', link)
print('working in', os.getcwd()); print('inputs:', os.listdir('data/processed'))
assert os.path.exists(f'{PROC}/predictor_table_1982_2025.csv'), 'PROC path is wrong: fix it in this cell'


## Scripts (written to the repository folder; add them to GitHub afterwards)

In [ ]:
%%writefile roni_ndj.csv
year,roni_ndj
1982,2.4
1983,-1.0
1984,-0.9
1985,-0.2
1986,1.4
1987,0.8
1988,-1.9
1989,0.1
1990,0.4
1991,2.0
1992,0.4
1993,0.4
1994,1.3
1995,-1.0
1996,-0.3
1997,2.3
1998,-1.6
1999,-1.6
2000,-0.8
2001,-0.3
2002,1.1
2003,0.2
2004,0.7
2005,-0.8
2006,0.9
2007,-1.6
2008,-0.9
2009,1.5
2010,-1.5
2011,-1.0
2012,-0.3
2013,-0.3
2014,0.6
2015,2.3
2016,-0.9
2017,-1.1
2018,0.8
2019,0.2
2020,-1.4
2021,-1.2
2022,-1.0
2023,1.4
2024,-1.0


In [ ]:
%%writefile roni_test.py
"""Previous-winter ENSO test with the Relative Oceanic Nino Index (RONI, ERSSTv6, NOAA CPC) in place of the ONI.
roni_ndj.csv: NDJ RONI by the year of November (from the CPC RONI table, accessed November 2026)."""
import numpy as np, pandas as pd
from scipy import stats
df = pd.read_csv('data/processed/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)
r = pd.read_csv('roni_ndj.csv').set_index('year').roni_ndj
YRS = np.arange(1983, 2026)
X = pd.Series(r.reindex(YRS - 1).values, index=YRS)
out = {}
for lt in ('mhw_detrended', 'mhw_raw'):
    lab = df[lt].astype(int); onset = (lab == 1) & (lab.shift(1, fill_value=0) == 0)
    for name, months in (('FebDec', range(2, 13)), ('MAMJ', range(3, 7))):
        m = lab.index.month.isin(list(months))
        days = lab[m].groupby(lab.index[m].year).sum().reindex(YRS, fill_value=0)
        ev = onset[m].groupby(lab.index[m].year).sum().reindex(YRS, fill_value=0)
        for resp, y in (('days', days), ('events', ev)):
            rho, p = stats.spearmanr(X, y)
            loo = [stats.spearmanr(np.delete(X.values, i), np.delete(y.values, i))[0] for i in range(len(YRS))]
            out[(lt, f'{resp}_{name}')] = (round(rho, 2), round(p, 3), f'{min(loo):+.2f} to {max(loo):+.2f}', round(stats.pearsonr(X, y)[0], 2))
T = pd.DataFrame(out, index=['Spearman', 'p', 'LOYO range', 'Pearson']).T
print(T.to_string())
lab = df.mhw_detrended.astype(int); m = lab.index.month >= 2
days = lab[m].groupby(lab.index[m].year).sum().reindex(YRS, fill_value=0)
ph = pd.cut(X, [-np.inf, -0.5, 0.5, np.inf], right=False, labels=['La Nina', 'Neutral', 'El Nino'])
print(days.groupby(ph, observed=True).agg(['mean', 'count']).round(1))
el = ph == 'El Nino'
print('Mann-Whitney one-sided p =', round(stats.mannwhitneyu(days[el], days[~el], alternative='greater').pvalue, 3))
top = days.nlargest(5)
print('top-5 years:', {int(y): (int(d), float(X[y]), str(ph[y])) for y, d in top.items()})


In [ ]:
%%writefile label_input_test.py
"""Label used as a model input: logistic SST model with and without the official Hobday label of the
forecast day as an extra predictor (detrended labels, OISST, leave-one-year-out, 1982-2025).
Run from the repository root:  python label_input_test.py [lead ...]"""
import os, sys, numpy as np, pandas as pd, warnings
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
warnings.filterwarnings('ignore')

df = pd.read_csv('data/processed/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)
OUT = 'results/label_input'; os.makedirs(OUT, exist_ok=True)
DATES = df.index

def doy366(idx):
    d = idx.dayofyear.values.copy(); d[(~idx.is_leap_year) & (idx.month > 2)] += 1
    return d

def hobday_mean(s, base=(1982, 2021), win=5, smooth=31):
    v, d = s.values, doy366(s.index)
    inb = (s.index.year >= base[0]) & (s.index.year <= base[1])
    c = np.full(367, np.nan)
    for k in range(1, 367):
        if k == 60: continue
        idx = np.where((d == k) & inb)[0]
        pool = np.unique(np.clip((idx[:, None] + np.arange(-win, win + 1)).ravel(), 0, len(v) - 1))
        c[k] = np.nanmean(v[pool[inb[pool]]])
    c[60] = (c[59] + c[61]) / 2; c = c[1:]
    c = np.convolve(np.r_[c[-(smooth // 2):], c, c[:smooth // 2]], np.ones(smooth) / smooth, 'valid')
    return pd.Series(c[d - 1], index=s.index)

def run_length(b):
    g = (~b).cumsum(); return b.astype(int).groupby(g).cumsum()

sst = df.sst_detrended; clim = hobday_mean(sst); thr = df.thresh90_detrended
a, dist = sst - clim, sst - thr
X = pd.DataFrame(index=DATES)
X['sst_anom'], X['sst_dist_thr'] = a, dist
for k in (1, 3, 7): X[f'lag{k}'] = a.shift(k)
for w in (3, 7, 14, 30): X[f'rm{w}'] = a.rolling(w).mean()
X['sd7'], X['max7'] = a.rolling(7).std(), a.rolling(7).max()
for k in (1, 3, 7): X[f'chg{k}'] = a - a.shift(k)
e = dist > 0
X['exceed_now'] = e.astype(int); X['exceed_run'] = run_length(e); X['mhw_causal'] = (X.exceed_run >= 5).astype(int)
doy = DATES.dayofyear
X['doy_sin'], X['doy_cos'] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
SST_COLS = list(X.columns)
X['label_t'] = df.mhw_detrended.astype(int)          # official Hobday label of the forecast day (uses later days)

def mcc(tp, fp, fn, tn):
    tp, fp, fn, tn = (np.asarray(v, float) for v in (tp, fp, fn, tn))
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0.0)

def best_threshold(y, p):
    u = np.unique(p)
    extra = u if len(u) <= 500 else np.quantile(p, np.linspace(0.5, 1, 301))
    th = np.unique(np.r_[np.arange(0.01, 1.0, 0.01), extra]); th = th[(th > 0) & (th < 1)]
    P = p[None, :] >= th[:, None]
    m = mcc((P & (y == 1)).sum(1), (P & (y == 0)).sum(1), ((~P) & (y == 1)).sum(1), ((~P) & (y == 0)).sum(1))
    return th[np.argmax(m)], m.max()

def train_mask(tab, test_years):
    m = ~tab.year.isin(test_years) & ~tab.target_year.isin(test_years)
    after = tab.year.isin([y + 1 for y in test_years]) & (tab.index.dayofyear <= 30)
    return m & ~after

logit = lambda C: make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight='balanced', max_iter=3000))

def loyo(tab, cols, inner_folds=4):
    out = []
    for y in sorted(tab.year.unique()):
        tr, te = tab[train_mask(tab, [y])], tab[tab.year == y]
        chunks = np.array_split(np.array(sorted(tr.year.unique())), inner_folds)
        best = None
        for C in (0.01, 0.1, 1.0):
            oof = pd.Series(np.nan, index=tr.index)
            for ch in chunks:
                itr, iva = tr[train_mask(tr, list(ch))], tr[tr.year.isin(ch)]
                oof.loc[iva.index] = logit(C).fit(itr[cols], itr.y).predict_proba(iva[cols])[:, 1]
            ok = oof.notna(); sc = best_threshold(tr.y[ok].values, oof[ok].values)[1]
            if best is None or sc > best[0]: best = (sc, C, oof[ok], tr.y[ok])
        _, C, oof, yo = best
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(oof.values, yo.values)
        th, _ = best_threshold(yo.values, iso.predict(oof.values))
        p = iso.predict(logit(C).fit(tr[cols], tr.y).predict_proba(te[cols])[:, 1])
        out.append(pd.Series((p >= th).astype(int), index=te.index))
    return pd.concat(out)

LEADS = [int(v) for v in sys.argv[1:] if v != 'score'] or ([] if sys.argv[1:] == ['score'] else [1, 3, 5, 7, 10, 14])
lab = df.mhw_detrended
for h in LEADS:
    if os.path.exists(f'{OUT}/lead{h}.csv'):                # resume after a Colab disconnect
        print('lead', h, 'already done'); continue
    tb = X.assign(y=lab.shift(-h), year=DATES.year, target_year=(DATES + pd.Timedelta(days=h)).year).dropna()
    tb['y'] = tb.y.astype(int)
    pd.DataFrame({'year': tb.year, 'y': tb.y, 'causal': tb.mhw_causal, 'exceed': tb.exceed_now, 'label': tb.label_t,
                  'sst': loyo(tb, SST_COLS), 'sst_label': loyo(tb, SST_COLS + ['label_t'])}).to_csv(f'{OUT}/lead{h}.csv')
    print('lead', h, 'done', flush=True)

# ---------- scores
YRS = list(range(1982, 2026)); IDX = np.random.default_rng(0).integers(0, len(YRS), (2000, len(YRS)))
def counts(p, y, yr):
    g = pd.DataFrame({'tp': (p == 1) & (y == 1), 'fp': (p == 1) & (y == 0), 'fn': (p == 0) & (y == 1), 'tn': (p == 0) & (y == 0), 'yr': yr}).groupby('yr').sum()
    return g.reindex(YRS, fill_value=0).values
pt = lambda c: float(mcc(*c.sum(0))); bt = lambda c: mcc(*c[IDX].sum(1).T)
ci = lambda d: f'{np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}'
rows = []
for h in (1, 3, 5, 7, 10, 14):
    f = f'{OUT}/lead{h}.csv'
    if not os.path.exists(f): continue
    o = pd.read_csv(f, index_col=0, parse_dates=True)
    C = {n: counts(o[n].values, o.y.values, o.year.values) for n in ('causal', 'exceed', 'label', 'sst', 'sst_label')}
    Bt = {n: bt(c) for n, c in C.items()}
    rows.append({'lead': h, 'MCC SST model': round(pt(C['sst']), 3), 'MCC SST + label': round(pt(C['sst_label']), 3),
                 'gain from label input': f"{pt(C['sst_label']) - pt(C['sst']):+.3f} ({ci(Bt['sst_label'] - Bt['sst'])})",
                 'label-based persistence': round(pt(C['label']), 3),
                 'SST + label minus label-based persistence': f"{pt(C['sst_label']) - pt(C['label']):+.3f} ({ci(Bt['sst_label'] - Bt['label'])})",
                 'SST + label minus causal persistence': f"{pt(C['sst_label']) - pt(C['causal']):+.3f} ({ci(Bt['sst_label'] - Bt['causal'])})"})
T = pd.DataFrame(rows); T.to_csv(f'{OUT}/label_input_test.csv', index=False)
pd.set_option('display.width', 300); pd.set_option('display.max_columns', 20); print(T.to_string(index=False))


In [ ]:
%%writefile past_only_paired_test.py
"""Past-only vs full-record reference, expanding-window forecasts 2002-2025 (detrended labels).
Rebuilt from data/processed/predictor_table_1982_2025.csv of the public repository."""
import sys, numpy as np, pandas as pd, warnings
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
warnings.filterwarnings('ignore')

# Run from the repository root: python past_only_paired_test.py
import os
df = pd.read_csv('data/processed/predictor_table_1982_2025.csv', index_col=0, parse_dates=True)
OUT = 'results/past_only'; os.makedirs(OUT, exist_ok=True)
DATES = df.index
TESTY = list(range(2002, 2026))
LEADS = [1, 3, 5, 7, 10, 14]

def doy366(idx):
    d = idx.dayofyear.values.copy(); d[(~idx.is_leap_year) & (idx.month > 2)] += 1
    return d
D366 = doy366(DATES)

def hobday_clim(v, base, win=5, smooth=31, pct=90):
    yrs = DATES.year.values
    inb = (yrs >= base[0]) & (yrs <= base[1])
    m, p = np.full(367, np.nan), np.full(367, np.nan)
    for k in range(1, 367):
        if k == 60: continue
        idx = np.where((D366 == k) & inb)[0]
        pool = np.unique(np.clip((idx[:, None] + np.arange(-win, win + 1)).ravel(), 0, len(v) - 1))
        x = v[pool[inb[pool]]]
        m[k], p[k] = np.nanmean(x), np.nanpercentile(x, pct)
    out = []
    for c in (m, p):
        c[60] = (c[59] + c[61]) / 2; c = c[1:]
        c = np.convolve(np.r_[c[-(smooth // 2):], c, c[:smooth // 2]], np.ones(smooth) / smooth, 'valid')
        out.append(c[D366 - 1])
    return out

def detect(exc, min_dur=5, max_gap=2):
    e = np.asarray(exc, int); n = len(e)
    b = np.diff(np.r_[0, e, 0]); st, en = np.where(b == 1)[0], np.where(b == -1)[0] - 1
    keep = en - st + 1 >= min_dur; st, en = st[keep], en[keep]
    causal, label = np.zeros(n, int), np.zeros(n, int)
    for s0, e0 in zip(st, en): label[s0:e0 + 1] = 1; causal[s0 + min_dur - 1:e0 + 1] = 1
    for k in range(len(st) - 1):
        if st[k + 1] - en[k] - 1 <= max_gap: label[en[k] + 1:st[k + 1]] = 1
    return causal, label

def run_length(b):
    g = (~b).cumsum(); return b.astype(int).groupby(g).cumsum()

def features(sd, clim, thr):
    a = pd.Series(sd - clim, index=DATES); dist = pd.Series(sd - thr, index=DATES)
    X = pd.DataFrame(index=DATES)
    X['sst_anom'], X['sst_dist_thr'] = a, dist
    for k in (1, 3, 7): X[f'lag{k}'] = a.shift(k)
    for w in (3, 7, 14, 30): X[f'rm{w}'] = a.rolling(w).mean()
    X['sd7'], X['max7'] = a.rolling(7).std(), a.rolling(7).max()
    for k in (1, 3, 7): X[f'chg{k}'] = a - a.shift(k)
    e = dist > 0
    X['exceed_now'] = e.astype(int); X['exceed_run'] = run_length(e); X['mhw_causal'] = (X.exceed_run >= 5).astype(int)
    doy = DATES.dayofyear
    X['doy_sin'], X['doy_cos'] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    return X

def reference(kind, Y):
    """SST series, climatology, threshold, labels for test year Y.
    kind: 'past'/'full' (detrended labels) or 'past_raw'/'full_raw' (raw labels)."""
    raw = df.sst.values.astype(float); t = np.arange(len(raw), dtype=float)
    if kind == 'full':
        sd = df.sst_detrended.values.astype(float); base = (1982, 2021)
    elif kind == 'full_raw':
        sd = raw; base = (1982, 2021)
    elif kind == 'past_raw':
        sd = raw; base = (1982, Y - 1)
    else:
        m = DATES.year.values <= Y - 1
        slope, icpt = np.polyfit(t[m], raw[m], 1)
        sd = raw - slope * t; base = (1982, Y - 1)
    clim, thr = hobday_clim(sd, base)
    causal, label = detect(sd > thr)
    return sd, clim, thr, causal, label

def mcc(tp, fp, fn, tn):
    tp, fp, fn, tn = (np.asarray(v, float) for v in (tp, fp, fn, tn))
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0.0)

def best_threshold(y, p):
    u = np.unique(p)
    extra = u if len(u) <= 500 else np.quantile(p, np.linspace(0.5, 1, 301))
    th = np.unique(np.r_[np.arange(0.01, 1.0, 0.01), extra]); th = th[(th > 0) & (th < 1)]
    P = p[None, :] >= th[:, None]
    m = mcc((P & (y == 1)).sum(1), (P & (y == 0)).sum(1), ((~P) & (y == 1)).sum(1), ((~P) & (y == 0)).sum(1))
    return th[np.argmax(m)], m.max()

def train_mask(tab, test_years):
    m = ~tab.year.isin(test_years) & ~tab.target_year.isin(test_years)
    after = tab.year.isin([y + 1 for y in test_years]) & (tab.index.dayofyear <= 30)
    return m & ~after

logit = lambda C: make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight='balanced', max_iter=3000))

def fit_predict(tr, te, cols, inner_folds=4):
    chunks = np.array_split(np.array(sorted(tr.year.unique())), inner_folds)
    best = None
    for C in (0.01, 0.1, 1.0):
        oof = pd.Series(np.nan, index=tr.index)
        for ch in chunks:
            itr, iva = tr[train_mask(tr, list(ch))], tr[tr.year.isin(ch)]
            oof.loc[iva.index] = logit(C).fit(itr[cols], itr.y).predict_proba(iva[cols])[:, 1]
        ok = oof.notna(); sc = best_threshold(tr.y[ok].values, oof[ok].values)[1]
        if best is None or sc > best[0]: best = (sc, C, oof[ok], tr.y[ok])
    _, C, oof, yo = best
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(oof.values, yo.values)
    th, _ = best_threshold(yo.values, iso.predict(oof.values))
    p = iso.predict(logit(C).fit(tr[cols], tr.y).predict_proba(te[cols])[:, 1])
    return (p >= th).astype(int)

import sys
KINDS = [] if sys.argv[1:] == ['score'] else (sys.argv[1:] or ['past', 'full', 'past_raw', 'full_raw'])
for kind in KINDS:
  for h in LEADS:
      if os.path.exists(f'{OUT}/{kind}_h{h}.csv'):        # resume after a Colab disconnect
          print(kind, h, 'already done'); continue
      rows = []
      for Y in TESTY:
          sd, clim, thr, causal, label = reference(kind, Y)
          X = features(sd, clim, thr); cols = list(X.columns)
          lab = pd.Series(label, index=DATES)
          tb = X.assign(y=lab.shift(-h), year=DATES.year, target_year=(DATES + pd.Timedelta(days=h)).year,
                        target=DATES + pd.Timedelta(days=h), label_t=label).dropna()
          tb['y'] = tb.y.astype(int)
          te = tb[tb.year == Y]
          tr = tb[tb.target <= pd.Timestamp(f'{Y}-01-01') - pd.Timedelta(days=10)]
          pred = fit_predict(tr, te, cols)
          rows.append(pd.DataFrame({'year': Y, 'y': te.y.values, 'model': pred, 'causal': te.mhw_causal.values,
                                    'exceed': te.exceed_now.values, 'label': te.label_t.values.astype(int),
                                    'label_day': lab.reindex(te.index).values}, index=te.index))
          print(kind, h, Y, flush=True)
      pd.concat(rows).to_csv(f'{OUT}/{kind}_h{h}.csv')

# ---------- scores, Table 6 and paired test (Table S13) ----------
B=2000; YRS=list(range(2002,2026)); IDX=np.random.default_rng(0).integers(0,len(YRS),(B,len(YRS)))
def mcc(tp,fp,fn,tn):
    tp,fp,fn,tn=(np.asarray(v,float) for v in (tp,fp,fn,tn)); den=np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return np.where(den>0,(tp*tn-fp*fn)/np.where(den>0,den,1),0.0)
def counts(p,y,yr):
    g=pd.DataFrame({'tp':(p==1)&(y==1),'fp':(p==1)&(y==0),'fn':(p==0)&(y==1),'tn':(p==0)&(y==0),'yr':yr}).groupby('yr').sum()
    return g.reindex(YRS,fill_value=0).values
pt=lambda c: float(mcc(*c.sum(0))); bt=lambda c: mcc(*c[IDX].sum(1).T)
ci=lambda d: f'{np.percentile(d,2.5):+.3f} to {np.percentile(d,97.5):+.3f}'
rows=[]
for lt,(KP,KF) in {'detrended':('past','full'),'raw':('past_raw','full_raw')}.items():
  if not all(os.path.exists(f'{OUT}/{k}_h1.csv') for k in (KP,KF)): continue
  for h in (1,3,5,7,10,14):
      R={}
      for k,kk in (('past',KP),('full',KF)):
          o=pd.read_csv(f'{OUT}/{kk}_h{h}.csv',index_col=0,parse_dates=True)
          C={n:counts(o[n].values,o.y.values,o.year.values) for n in ('model','causal','exceed','label')}
          R[k]=(C,{n:bt(c) for n,c in C.items()},o)
      Cp,Bp,op=R['past']; Cf,Bf,of=R['full']
      gp=Bp['model']-Bp['exceed']; gf=Bf['model']-Bf['exceed']
      gpc=Bp['model']-Bp['causal']; gfc=Bf['model']-Bf['causal']
      rows.append({'labels':lt,'lead':h,'causal':round(pt(Cp['causal']),3),'exceed':round(pt(Cp['exceed']),3),'logit':round(pt(Cp['model']),3),
        'inflation':f"{pt(Cp['label'])-pt(Cp['causal']):+.3f} ({ci(Bp['label']-Bp['causal'])})",
        'gain_causal_past':f"{pt(Cp['model'])-pt(Cp['causal']):+.3f} ({ci(gpc)})",
        'gain_exc_past':f"{pt(Cp['model'])-pt(Cp['exceed']):+.3f} ({ci(gp)})",
        'gain_exc_full':f"{pt(Cf['model'])-pt(Cf['exceed']):+.3f} ({ci(gf)})",
        'DIFF_exc past-full':f"{(pt(Cp['model'])-pt(Cp['exceed']))-(pt(Cf['model'])-pt(Cf['exceed'])):+.3f} ({ci(gp-gf)})",
        'DIFF_causal past-full':f"{(pt(Cp['model'])-pt(Cp['causal']))-(pt(Cf['model'])-pt(Cf['causal'])):+.3f} ({ci(gpc-gfc)})"})
T=pd.DataFrame(rows); T.to_csv(f'{OUT}/paired_test.csv',index=False)
pd.set_option('display.width',300); pd.set_option('display.max_columns',20); print(T.to_string(index=False))
o1p=pd.read_csv(f'{OUT}/past_h1.csv',index_col=0,parse_dates=True); o1f=pd.read_csv(f'{OUT}/full_h1.csv',index_col=0,parse_dates=True)
print('MHW days 2002-2025 past-only:',int(o1p.label_day.sum()),'full:',int(o1f.label_day.sum()),
      'label MCC:',round(float(mcc(*counts(o1p.label_day.values,o1f.label_day.values,o1p.year.values).sum(0))),3))


In [ ]:
%%writefile make_figures.py
"""Manuscript Figs 2, 4, 6, 7 and supplementary Fig. S3, redrawn from the result tables in results/tables.
Run from the repository root:  python make_figures.py   ->  results/figures/manuscript/*.png (600 dpi)"""
import os, re, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

T = 'results/tables'; OUT = 'results/figures/manuscript'; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8,
                     'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5, 'legend.fontsize': 7, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
                     'grid.color': '#e6e6e6', 'grid.linewidth': 0.5, 'savefig.dpi': 600, 'savefig.bbox': 'tight'})
# one colour per entity across all figures (validated: lightness, chroma, CVD and normal-vision separation)
C = dict(causal='#4b5563', exceed='#1a9e77', logit='#2a76d2', rf='#e8601c', label='#b0479a',
         oisst='#2a76d2', cci='#e8601c', bob='#2a76d2', lev='#e8601c', scs='#1a9e77')
M = dict(causal='D', exceed='v', logit='o', rf='^', label=None)
LEADS = [1, 3, 5, 7, 10, 14]

def rng(s):
    a, b = re.findall(r'[+-−]?\d*\.\d+', str(s).replace('−', '-'))[:2]
    return float(a), float(b)

def style(ax, leads):
    ax.set_xticks(leads); ax.grid(True); ax.set_axisbelow(True); ax.set_xlabel('Lead (days)')

# ---------------- Fig. 2: schematic of the three benchmarks
def fig2():
    d = np.r_[-0.40, -0.30, -0.20, -0.10, 0.15, 0.25, 0.35, 0.45, 0.50, 0.55, 0.45, 0.30, 0.20, 0.10,
              -0.15, -0.30, -0.20, -0.10, 0.10, 0.15, -0.10, -0.30, -0.35, -0.30, -0.40, -0.45]
    days = np.arange(1, 27)
    fig, (a, b) = plt.subplots(2, 1, figsize=(6.3, 3.6), gridspec_kw=dict(height_ratios=[1.25, 1], hspace=0.15), sharex=True)
    a.axvspan(4.5, 8.5, color=C['label'], alpha=0.10, lw=0)
    a.text(6.5, 0.83, 'known only\nin hindsight', ha='center', va='top', color='#7a2f69', fontsize=7)
    a.bar(days, d, width=0.8, color=np.where(d > 0, C['rf'], '#a3aab4'), edgecolor='white', linewidth=0.5)
    a.axhline(0, color='#333333', lw=0.7)
    a.text(19.5, 0.42, '2-day exceedance:\nnever an MHW', ha='center', fontsize=7, color='#333333')
    a.set_ylabel('SST minus\nthreshold (°C)'); a.set_ylim(-0.52, 0.88); a.grid(True, axis='y'); a.set_axisbelow(True)
    rows = [('Official Hobday label', C['label'], list(range(5, 15))),
            ('Causal persistence state', C['causal'], list(range(9, 15))),
            ('Exceedance on the day', C['exceed'], list(range(5, 15)) + [19, 20])]
    for k, (name, col, on) in enumerate(rows):
        y = 2 - k
        b.bar(on, 0.62, bottom=y - 0.31, width=0.82, color=col, edgecolor='white', linewidth=0.6)
    b.set_yticks([2, 1, 0]); b.set_yticklabels([r[0] for r in rows]); b.set_ylim(-0.6, 2.6)
    b.tick_params(axis='y', length=0); b.spines['left'].set_visible(False)
    b.set_xticks([1, 5, 9, 14, 20, 26]); b.set_xlim(0.3, 26.7); b.set_xlabel('Day')
    fig.savefig(f'{OUT}/fig2_benchmark_schematic.png'); plt.close(fig)

# ---------------- Fig. 4 and Fig. S3: inflation curves
def fig4():
    d = pd.read_csv(f'{T}/wp1_leakage_curve_3regions.csv')
    fig, axs = plt.subplots(1, 2, figsize=(6.3, 2.6), sharey=True)
    regs = [('Bay of Bengal', 'bob', 'o'), ('Levantine Sea', 'lev', 's'), ('South China Sea', 'scs', '^')]
    for ax, (tg, title) in zip(axs, (('det', '(a) Detrended labels'), ('raw', '(b) Raw labels'))):
        for reg, key, mk in regs:
            s = d[(d.region == reg) & (d.target == tg)].sort_values('lead')
            ax.fill_between(s.lead, s.inflation_total_lo, s.inflation_total_hi, color=C[key], alpha=0.13, lw=0)
            ax.plot(s.lead, s.inflation_total, color=C[key], marker=mk, ms=4, lw=1.4, label=reg)
        ax.axhline(0, color='#333333', lw=0.6); style(ax, [1, 2, 3, 5, 7, 10, 14]); ax.set_title(title)
    axs[0].set_ylabel('MCC inflation\n(label-based minus causal)')
    axs[0].legend(frameon=False, loc='upper right')
    fig.savefig(f'{OUT}/fig4_inflation_three_regions.png'); plt.close(fig)

def figS3():
    d = pd.read_csv(f'{T}/wp1c_grid_leakage_summary.csv')
    fig, axs = plt.subplots(1, 2, figsize=(6.3, 2.6), sharey=True)
    for ax, (tg, title) in zip(axs, (('det', '(a) Detrended labels'), ('raw', '(b) Raw labels'))):
        s = d[d.labels == tg].sort_values('lead')
        lo, hi = zip(*[rng(v) for v in s['cell inflation 10th-90th pct']])
        ax.fill_between(s.lead, lo, hi, color=C['logit'], alpha=0.18, lw=0, label='Grid cells, 10th–90th percentile')
        ax.plot(s.lead, s['cell inflation median'], color=C['logit'], marker='o', ms=4, lw=1.4, label='Grid cells, median')
        ax.plot(s.lead, s['box-mean inflation'], color=C['causal'], marker='s', ms=4, lw=1.2, ls='--', label='Regional mean')
        ax.axhline(0, color='#333333', lw=0.6); style(ax, [1, 2, 3, 5, 7, 10, 14]); ax.set_title(title)
    axs[0].set_ylabel('MCC inflation\n(label-based minus causal)'); axs[1].legend(frameon=False, loc='upper right')
    fig.savefig(f'{OUT}/figS3_grid_vs_regional_inflation.png'); plt.close(fig)

# ---------------- Fig. 6: skill by lead
def fig6():
    sk = pd.read_csv(f'{T}/wp2_skill_by_lead_det.csv'); ex = pd.read_csv(f'{T}/wp2_exceedance_persistence_det.csv')
    lb = pd.read_csv(f'{T}/wp1_leakage_curve_bob.csv'); lb = lb[(lb.target == 'det') & lb.lead.isin(LEADS)].sort_values('lead')
    g = lambda m: sk[sk.model == m].sort_values('lead')
    ex = ex.sort_values('lead')
    fig, (a, b) = plt.subplots(1, 2, figsize=(6.3, 3.2), gridspec_kw=dict(wspace=0.42))
    series = [('Causal persistence', 'causal', g('persistence').MCC, g('persistence').MCC_lo, g('persistence').MCC_hi),
              ('Exceedance persistence', 'exceed', ex['exceedance persistence'], *zip(*[rng(v) for v in ex['exc. pers. 95% CI']])),
              ('Logistic regression, SST', 'logit', g('logit_sst').MCC, g('logit_sst').MCC_lo, g('logit_sst').MCC_hi),
              ('Random forest, SST + ERA5', 'rf', g('rf_era5').MCC, g('rf_era5').MCC_lo, g('rf_era5').MCC_hi)]
    for name, k, y, lo, hi in series:
        a.fill_between(LEADS, lo, hi, color=C[k], alpha=0.10, lw=0)
        a.plot(LEADS, y, color=C[k], marker=M[k], ms=4, lw=1.4, label=name)
    a.plot(LEADS, lb.MCC_label, color=C['label'], ls=':', lw=1.8, label='Label-based persistence')
    style(a, LEADS); a.set_ylabel('MCC'); a.set_title('(a) Skill by lead'); a.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, columnspacing=1.0, handlelength=1.8)
    lsst = g('logit_sst').MCC.values
    for k, base, cis, dx, mk, name in (('causal', g('persistence').MCC.values, g('logit_sst')['gain vs persistence'], -0.18, 'D', 'Over causal persistence'),
                                       ('exceed', ex['exceedance persistence'].values, ex['logit_sst gain vs exc. pers.'], 0.18, 'v', 'Over exceedance persistence')):
        lo, hi = map(np.array, zip(*[rng(v) for v in cis])); p = lsst - base
        b.errorbar(np.array(LEADS) + dx, p, yerr=[p - lo, hi - p], fmt=mk, ms=4.5, color=C[k], ecolor=C[k], elinewidth=1.1, capsize=0, label=name)
    b.axhline(0, color='#333333', lw=0.7); style(b, LEADS); b.set_ylabel('MCC gain of the logistic SST model\n(95% CI)')
    b.set_title('(b) Gain of the logistic SST model'); b.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)
    fig.savefig(f'{OUT}/fig6_skill_by_lead.png'); plt.close(fig)

# ---------------- Fig. 7: OISST vs CCI/C3S
def fig7():
    mo = pd.read_csv(f'{T}/v2_diag_month.csv', index_col=0); lk = pd.read_csv(f'{T}/v2_leakage_both.csv'); sk = pd.read_csv(f'{T}/v2_skill_both.csv')
    fig, axs = plt.subplots(2, 2, figsize=(6.3, 5.8), gridspec_kw=dict(hspace=0.62, wspace=0.32)); (a, b), (c, d) = axs
    m = np.arange(1, 13)
    a.axvspan(5.5, 9.5, color='#eef0f3', lw=0, zorder=0)
    a.bar(m, mo.both / mo['MHW days OISST'], width=0.7, color='#a3aab4', label='OISST MHW days that are\nalso MHW days in CCI/C3S', zorder=2)
    a.plot(m, mo.r, color='#222222', marker='o', ms=3.5, lw=1.3, label='Correlation of daily anomalies', zorder=3)
    a.set_xticks(m); a.set_xticklabels(list('JFMAMJJASOND')); a.set_yticks(np.arange(0, 1.01, 0.2))
    a.set_xlabel('Calendar month'); a.set_ylabel('Fraction / correlation'); a.set_title('(a) Agreement between the products')
    a.text(7.5, 0.95, 'monsoon', ha='center', va='top', fontsize=6.5, color='#555555', zorder=4)
    a.set_ylim(0, 1.0); a.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=6.5, handlelength=1.4); a.grid(True, axis='y'); a.set_axisbelow(True)
    for ds, key, ls, mk in (('OISST', 'oisst', '-', 'o'), ('CCI', 'cci', '--', 's')):
        s = lk[lk.dataset == ds].sort_values('lead')
        b.fill_between(s.lead, s.lo, s.hi, color=C[key], alpha=0.13, lw=0)
        b.plot(s.lead, s.inflation, color=C[key], ls=ls, marker=mk, ms=3.5, lw=1.4, label='OISST' if ds == 'OISST' else 'CCI/C3S')
    b.axhline(0, color='#333333', lw=0.6); style(b, [1, 2, 3, 5, 7, 10, 14]); b.set_ylabel('MCC inflation')
    b.set_title('(b) Label-based minus causal persistence'); b.legend(frameon=False, loc='upper right')
    for ds, ls in (('OISST', '-'), ('CCI', '--')):
        s = sk[sk.dataset == ds].sort_values('lead')
        for col, k in (('honest', 'causal'), ('exceedance', 'exceed'), ('logistic', 'logit')):
            c.plot(s.lead, s[col], color=C[k], ls=ls, marker=M[k], ms=3.5, lw=1.3)
    style(c, LEADS); c.set_ylabel('MCC'); c.set_title('(c) Skill: solid OISST, dashed CCI/C3S')
    c.legend(handles=[Line2D([], [], color=C['causal'], marker='D', ms=3.5, label='Causal persistence'),
                      Line2D([], [], color=C['exceed'], marker='v', ms=3.5, label='Exceedance persistence'),
                      Line2D([], [], color=C['logit'], marker='o', ms=3.5, label='Logistic, SST'),
                      Line2D([], [], color='#555555', ls='-', label='OISST'), Line2D([], [], color='#555555', ls='--', label='CCI/C3S')],
             frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=3, columnspacing=0.9, handlelength=1.8, fontsize=6.5)
    for ds, key, dx, mk in (('OISST', 'oisst', -0.2, 'o'), ('CCI', 'cci', 0.2, 's')):
        s = sk[sk.dataset == ds].sort_values('lead')
        lo, hi = map(np.array, zip(*[rng(v) for v in s['logistic vs honest']])); p = (s.logistic - s.honest).values
        d.errorbar(s.lead + dx, p, yerr=[p - lo, hi - p], fmt=mk, ms=4, color=C[key], ecolor=C[key], elinewidth=1.1, capsize=0,
                   label='OISST' if ds == 'OISST' else 'CCI/C3S')
    d.axhline(0, color='#333333', lw=0.7); style(d, LEADS); d.set_ylabel('MCC gain'); d.set_title('(d) Logistic SST minus causal persistence')
    d.legend(frameon=False, loc='upper right')
    fig.savefig(f'{OUT}/fig7_oisst_vs_cci.png'); plt.close(fig)

for f in (fig2, fig4, figS3, fig6, fig7): f()
print('written to', OUT)


In [ ]:
%%writefile figS2_case_studies.py
"""Supplementary Fig. S2: forecasts for the three strongest detrended MHWs at 7- and 14-day leads.
Same event rule and data as notebook cell WP5-2 (largest cumulative intensity, 1982-2025); only the
figure is changed: 'Causal' spelling, readable date ticks, and the colours used in the other figures.
Usage:  python figS2_case_studies.py <PROC folder on Drive> <output folder>"""
import os, sys, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.dates as mdates

PROC, OUT = sys.argv[1], sys.argv[2]; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8,
                     'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5, 'legend.fontsize': 7, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.6, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.5,
                     'savefig.dpi': 600, 'savefig.bbox': 'tight'})
C = dict(causal='#4b5563', exceed='#1a9e77', logit='#2a76d2', obs='#e8601c')   # same colours as Figs 2, 4, 6, 7

rd = lambda f: pd.read_csv(f'{PROC}/{f}', index_col=0, parse_dates=True)
pt = rd('predictor_table_1982_2025.csv')
lab = pt.mhw_detrended.astype(int)
exc = (pt.sst_detrended > pt.thresh90_detrended).astype(int)
anom = rd('features_det_t+7.csv').sst_anom.reindex(lab.index)
# lead 7 = Phase 5 runs (oof_p5), lead 14 = WP2 runs (oof_lead14), exactly as in WP2-1
SRC = {7: 'oof_p5', 14: 'oof_lead14'}
RES = {h: {n: rd(f'{d}/{n}_det.csv') for n in ('logit_sst', 'persistence')} for h, d in SRC.items()}

# 1) events; selection rule fixed before looking at forecasts: largest cumulative intensity
b = lab.values; d = np.diff(np.r_[0, b, 0])
st, en = np.where(d == 1)[0], np.where(d == -1)[0] - 1
EV = pd.DataFrame({'onset': lab.index[st], 'end': lab.index[en], 'duration_d': en - st + 1,
                   'cum_intensity_Cdays': [anom.iloc[s:e + 1].sum() for s, e in zip(st, en)]})
TOP = EV.sort_values('cum_intensity_Cdays', ascending=False).head(3).sort_values('onset').reset_index(drop=True)
print(TOP.assign(cum_intensity_Cdays=TOP.cum_intensity_Cdays.round(1)).to_string(), '\n')

# 2) forecasts placed on their target date (issue date + lead)
def at_target(s, h):
    s = s.copy(); s.index = s.index + pd.Timedelta(days=h); return s
FC = {}
for h in SRC:
    m = RES[h]['logit_sst']
    FC[h] = {'prob': at_target(m.prob, h), 'thr': at_target(m.thr, h), 'logistic (SST)': at_target(m.pred, h),
             'exceedance persistence': at_target(exc.reindex(m.index), h),
             'causal persistence': at_target(RES[h]['persistence'].pred, h)}

# 3) per-event scores (Table S7)
day = pd.Timedelta(days=1); rows = []
for _, ev in TOP.iterrows():
    evd = pd.date_range(ev.onset, ev.end); pre = pd.date_range(ev.onset - 30 * day, ev.onset - day)
    quiet_pre = pre[lab.reindex(pre).fillna(1).values == 0]
    for h in SRC:
        for n in ('logistic (SST)', 'exceedance persistence', 'causal persistence'):
            s_ev = FC[h][n].reindex(evd).fillna(0); first = s_ev[s_ev == 1].index.min()
            rows.append({'event onset': f'{ev.onset:%Y-%m-%d}', 'lead': h, 'forecast': n,
                         'first event day caught (days after onset)': (first - ev.onset).days if pd.notna(first) else 'never',
                         'event days forecast (%)': int(round(100 * s_ev.mean())),
                         'false alarms, 30 d before onset': int(FC[h][n].reindex(quiet_pre).fillna(0).sum())})
CS = pd.DataFrame(rows); CS.to_csv(f'{OUT}/wp5_case_studies.csv', index=False)
print(CS.to_string(index=False))

# 4) figure: 3 events x 2 leads
fig, axs = plt.subplots(3, 2, figsize=(7.2, 7.4), sharey=True)
for i, ev in TOP.iterrows():
    win = pd.date_range(ev.onset - 30 * day, ev.end + 15 * day)
    obs = lab.reindex(win).fillna(0).values
    for j, h in enumerate(SRC):
        ax = axs[i, j]
        ax.fill_between(win, 0, 1, where=obs == 1, color=C['obs'], alpha=0.18, lw=0, step='mid', label='Observed MHW')
        ax.plot(win, FC[h]['prob'].reindex(win), color=C['logit'], lw=1.6, label='Logistic SST model, probability')
        ax.step(win, FC[h]['thr'].reindex(win), where='mid', color=C['logit'], ls='--', lw=0.9, label='Decision threshold')
        for k, (n, c, lb) in enumerate((('exceedance persistence', C['exceed'], 'Exceedance persistence: yes'),
                                        ('causal persistence', C['causal'], 'Causal persistence: yes'))):
            yes = (FC[h][n].reindex(win).fillna(0) == 1).values
            ax.plot(win[yes], np.full(yes.sum(), -0.07 - 0.07 * k), '|', color=c, ms=6, mew=1.2, label=lb)
        ax.set_ylim(-0.2, 1.03); ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
        ax.set_xlim(win[0], win[-1])
        ax.set_title(f'({"abcdef"[2 * i + j]}) {ev.onset:%b %Y} event, {h}-day lead', loc='left')
        # one labelled tick on the 1st of each month, unlabelled ticks on the 15th: no overlapping dates
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonthday=15))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('1 %b\n%Y'))
        ax.grid(True, which='major'); ax.grid(True, which='minor', color='#f0f0f0', lw=0.4)
    axs[i, 0].set_ylabel('Forecast probability')
hh, ll = axs[0, 0].get_legend_handles_labels()
fig.legend(hh, ll, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.045))
fig.tight_layout(h_pad=1.2)
fig.savefig(f'{OUT}/figS2_case_studies.png'); print('\nwritten', f'{OUT}/figS2_case_studies.png')


## 1. RONI previous-winter test (seconds)

In [ ]:
!python roni_test.py | tee "{REV}/roni_test_output.txt"

## 2. Label used as a model input (about 5 min; resumes per lead)

In [ ]:
!python label_input_test.py

## 3. Past-only vs full-record design, detrended and raw labels (about 5 min; resumes per design and lead)
Four designs × six leads × 24 test years. Progress is printed for every test year.

In [ ]:
!python past_only_paired_test.py

## 4. Corrected figures from the repository tables

In [ ]:
!python make_figures.py

## 5. Fig. S2 and Table S7 (case studies) from your Drive data

In [ ]:
!python figS2_case_studies.py "{PROC}" results/figures/manuscript

## 6. Check against the values in the manuscript
Every row should say `ok`. A difference of 0.001–0.003 can come from library versions; anything larger means something changed.

In [ ]:
import pandas as pd
def num(s): return float(str(s).split()[0])
li = pd.read_csv('results/label_input/label_input_test.csv').set_index('lead')
pt = pd.read_csv('results/past_only/paired_test.csv')
checks = [('label input gain, 1 d', num(li.loc[1, 'gain from label input']), 0.117),
          ('label input gain, 3 d', num(li.loc[3, 'gain from label input']), 0.093),
          ('label input gain, 5 d', num(li.loc[5, 'gain from label input']), 0.053),
          ('label input gain, 7 d', num(li.loc[7, 'gain from label input']), 0.028)]
EXP = {('detrended', 1): 0.018, ('detrended', 3): 0.012, ('detrended', 5): -0.025, ('detrended', 7): -0.022,
       ('detrended', 10): -0.055, ('detrended', 14): -0.065, ('raw', 1): 0.007, ('raw', 3): -0.010,
       ('raw', 5): -0.020, ('raw', 7): -0.029, ('raw', 10): -0.054, ('raw', 14): -0.036}
for _, r in pt.iterrows():
    checks.append((f"past-only minus full-record gain over exceedance, {r['labels']} {r['lead']} d",
                   num(r['DIFF_exc past-full']), EXP[(r['labels'], r['lead'])]))
for name, got, exp in checks:
    print(f"{'ok  ' if abs(got - exp) <= 0.003 else 'DIFF'} {name}: yours {got:+.3f}, manuscript {exp:+.3f}")
print('\nRONI: the manuscript reports rho = 0.36, p = 0.018 (detrended, MHW days Feb-Dec); compare with step 1 output.')


## 7. Copy everything to Drive and download a zip

In [ ]:
import shutil, glob
from google.colab import files
os.makedirs(f'{REV}/figures', exist_ok=True); os.makedirs(f'{REV}/scripts', exist_ok=True)
for f in glob.glob('results/figures/manuscript/*'): shutil.copy(f, f'{REV}/figures/')
for f in ('roni_ndj.csv', 'roni_test.py', 'label_input_test.py', 'past_only_paired_test.py',
          'make_figures.py', 'figS2_case_studies.py'): shutil.copy(f, f'{REV}/scripts/')
shutil.make_archive('/content/mhw_revision', 'zip', REV, '.')
print('saved to', REV); files.download('/content/mhw_revision.zip')
